# Streaming chat + conversation threading

Two things every chat UI needs:

1. **Streaming** — consume Server-Sent Events (SSE) token-by-token instead of waiting for the whole answer.
2. **Threading** — keep context across turns with `previous_response_id`.

Aura implements the Open Responses API, so streaming is a sequence of typed events (`response.output_text.delta`, `response.completed`, ...) rather than raw token deltas.

In [ ]:
from aura import AuraClient
client = AuraClient()

In [ ]:
# 1. Stream a response and print text deltas as they arrive
stream = client.responses.create(
    model="gpt-5.4-mini",
    input="Count from 1 to 5, one number per line.",
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        print(f"\n\n[completed] status={event.response.status}")

In [ ]:
# 2. Conversation threading with previous_response_id
turn1 = client.responses.create(
    model="gpt-5.4-mini",
    input="My favorite color is teal. Remember that.",
)
print(f"turn 1: {turn1.output_text}")

turn2 = client.responses.create(
    model="gpt-5.4-mini",
    input="What is my favorite color?",
    previous_response_id=turn1.id,
)
print(f"turn 2: {turn2.output_text}")

In [ ]:
# 3. Streaming + threading together (stateful streamed chat)
history_id = None
for question in ["What's 3 * 7?", "And what was the first question I asked?"]:
    print(f"\n> {question}")
    events = client.responses.create(
        model="gpt-5.4-mini",
        input=question,
        previous_response_id=history_id,
        stream=True,
    )
    for event in events:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
        elif event.type == "response.completed":
            history_id = event.response.id
            print()